# MP2: Data Cleaning & Feature Engineering

**MajsterPlus — Customer Lapse Prediction**

In this mini-project, you will follow a **mandatory 12-step preprocessing pipeline**
to clean the data and prepare it for modeling. The steps must be executed in order.

**CRISP-DM Phase**: Data Preparation

---

## 0. Setup & Reproducibility

In [1]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Library version assertions
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — expected 1.4.x or 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — expected 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — expected <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

sklearn 1.5.2, pandas 2.3.3, numpy 1.26.4
Random seed: 42


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## Step 1–2: Load Data & Verify Fingerprint

In [3]:
import hashlib
from pathlib import Path

# Colab detection
try:
    import google.colab
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
except ImportError:
    DATA_DIR = Path("../2. data")

assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

# Load and verify customers.csv
customers = pd.read_csv(DATA_DIR / "customers.csv")
cust_md5 = hashlib.md5((DATA_DIR / "customers.csv").read_bytes()).hexdigest()
assert cust_md5 == "c016cd4bfc36ac8d0a99bb6a3d55fa44", (
    f"customers.csv MD5 mismatch: {cust_md5} != c016cd4bfc36ac8d0a99bb6a3d55fa44"
)
print(f"customers.csv loaded: {customers.shape}, MD5 OK")

# Load and verify transactions.csv
txn_md5 = hashlib.md5((DATA_DIR / "transactions.csv").read_bytes()).hexdigest()
assert txn_md5 == "a9c4bcfc4878baae6f5c250d4d15d737", (
    f"transactions.csv MD5 mismatch: {txn_md5} != a9c4bcfc4878baae6f5c250d4d15d737"
)
print(f"transactions.csv verified: MD5 OK")

customers.csv loaded: (5000, 21), MD5 OK
transactions.csv verified: MD5 OK


## Step 3: Separate Target & Drop ID

In [4]:
# Save target before any transformations
y = customers["is_lapsed"].copy()
print(f"Target saved: {y.shape[0]} values, lapse rate = {y.mean():.3f}")

# Drop customer_id (not a feature) and is_lapsed (target)
# Keep gender for later fairness analysis
gender_series = customers["gender"].copy()

df = customers.drop(columns=["customer_id", "is_lapsed"])
print(f"Working DataFrame: {df.shape}")

Target saved: 5000 values, lapse rate = 0.195
Working DataFrame: (5000, 19)


## Step 4: Parse Polish Dates

The `registration_date` column uses Polish month abbreviations (e.g., "15-sty-2024").
We provide the parsing helper — your task is to apply it and verify the derived feature.

In [5]:
# Helper: parse Polish date strings into datetime
POLISH_MONTHS = {
    "sty": 1, "lut": 2, "mar": 3, "kwi": 4, "maj": 5, "cze": 6,
    "lip": 7, "sie": 8, "wrz": 9, "pa\u017a": 10, "lis": 11, "gru": 12,
}

def parse_polish_date(date_str: str) -> pd.Timestamp:
    """Convert Polish date string 'DD-mmm-YYYY' to Timestamp."""
    day, month_str, year = date_str.split("-")
    return pd.Timestamp(year=int(year), month=POLISH_MONTHS[month_str], day=int(day))

In [6]:
# Apply the parse_polish_date helper
df["registration_date"] = df["registration_date"].apply(parse_polish_date)
df = df.drop(columns=["registration_date"])

## Step 5: Clean total_spend

The `total_spend` column contains currency-formatted strings like `"PLN 1,234.50"`.
Convert it to a numeric float column.

In [7]:
df["total_spend"] = df["total_spend"].str.replace("PLN ", "").str.replace(",", "").astype(float)
print(df["total_spend"].describe())

count     5000.000000
mean      1566.364156
std       2044.618020
min         35.000000
25%        232.112500
50%        965.095000
75%       2044.240000
max      26832.420000
Name: total_spend, dtype: float64


### Interpretation: total_spend distribution

**TODO**: Compare the mean and median of `total_spend` (use the `.describe()` output above). What does the discrepancy between these two measures tell you about the distribution shape? Is it right-skewed, left-skewed, or approximately normal? Why might this matter for modeling?

In [8]:
print(f"Mean: {df['total_spend'].mean()}")
print(f"Median: {df['total_spend'].median()}")
# Distribution is right-skewed

Mean: 1566.364156
Median: 965.095


## Step 6: Handle Impossible satisfaction_score Values

Valid range for `satisfaction_score` is [1.0, 5.0]. Any values outside this range
are data entry errors and should be replaced with NaN.

### Think About This

Why must impossible values be replaced with NaN **before** running imputation in Step 7? What would happen to the median if the impossible values (e.g., 0.0, 7.2, -1.0) were included in the computation?

In [9]:
imp = (df["satisfaction_score"] < 1.0) | (df["satisfaction_score"] > 5.0)
print("Impossible scores found:", imp.sum())
df.loc[imp, "satisfaction_score"] = float('nan')

Impossible scores found: 3


## Step 7: Impute Missing Values

After Steps 4–6, several columns have missing values. Apply appropriate imputation:
- **Numeric columns** (age, online_ratio, satisfaction_score): median imputation
- **Categorical columns** (monthly_income_bracket, referral_source): mode imputation

In [10]:
for col in ["age", "online_ratio", "satisfaction_score"]:
    df[col] = df[col].fillna(df[col].median())
for col in ["monthly_income_bracket", "referral_source"]:
    df[col] = df[col].fillna(df[col].mode()[0])
df["age"] = df["age"].astype(int)
print(df.isnull().sum())

age                          0
gender                       0
city                         0
loyalty_member               0
total_spend                  0
purchase_count               0
avg_basket_value             0
days_since_last_purchase     0
product_categories_bought    0
online_ratio                 0
satisfaction_score           0
customer_service_contacts    0
newsletter_subscriber        0
monthly_income_bracket       0
district_type                0
store_distance_km            0
referral_source              0
account_age_days             0
dtype: int64


## Step 8: Remove IQR Outliers

The `avg_basket_value` column has extreme outliers. Use the IQR method
with factor=1.5 to identify and remove them.

In [11]:
# IQR calculation (pre-filled)
Q1 = df["avg_basket_value"].quantile(0.25)
Q3 = df["avg_basket_value"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")

IQR bounds: [-289.70, 732.07]


In [12]:
mask = (df["avg_basket_value"] >= lower_bound) & (df["avg_basket_value"] <= upper_bound)
print("Rows removed:", (~mask).sum())
df = df[mask]
y = y[mask]
gender_series = gender_series[mask]

Rows removed: 22


## Step 9: Encode Categorical Variables

Encode all categorical columns in the following order:
1. **Binary**: `loyalty_member` (Tak/Nie)
2. **Ordinal**: `monthly_income_bracket` (A through E, representing increasing income)
3. **One-hot**: remaining categorical columns with `pd.get_dummies(drop_first=False)`
4. Sort columns alphabetically and convert any boolean dummy columns to int.

In [13]:
df["loyalty_member"] = df["loyalty_member"].map({"Tak": 1, "Nie": 0})
df["monthly_income_bracket"] = df["monthly_income_bracket"].map({"A": 1, "B": 2, "C": 3, "D": 4, "E": 5})

df = pd.get_dummies(df, drop_first=False)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
df = df[sorted(df.columns)]
print(df.shape)
print(df.dtypes)

(4978, 37)
account_age_days                       int64
age                                    int64
avg_basket_value                     float64
city_Białystok                         int64
city_Bydgoszcz                         int64
city_Gdańsk                            int64
city_Gdynia                            int64
city_Katowice                          int64
city_Kraków                            int64
city_Lublin                            int64
city_Poznań                            int64
city_Szczecin                          int64
city_Warszawa                          int64
city_Wrocław                           int64
city_Łódź                              int64
customer_service_contacts              int64
days_since_last_purchase               int64
district_type_rural                    int64
district_type_suburban                 int64
district_type_urban                    int64
gender_K                               int64
gender_M                               int64

### Interpretation: drop_first trade-off

**TODO**: The course uses `drop_first=False`. What would happen if you used `drop_first=True` instead? Which types of models would benefit from dropping the first category, and why? Which types of models would be unaffected?

In [14]:
# `drop_first=True` avoids perfect multicollinearity,
# which is needed for linear models. For tree-based models, drop_first=False is fine.

## Step 10: Null Assertion Gate

In [15]:
# This MUST pass before proceeding
assert df.isnull().sum().sum() == 0, (
    f"Still have {df.isnull().sum().sum()} null values! Fix Steps 6-7."
)
print(f"Null assertion passed. DataFrame shape: {df.shape}")
print(f"All dtypes numeric: {all(df.dtypes.apply(lambda x: np.issubdtype(x, np.number)))}")

Null assertion passed. DataFrame shape: (4978, 37)
All dtypes numeric: True


## Step 11: Train/Test Split

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42, stratify=y
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train lapse rate: {y_train.mean():.3f}")
print(f"y_test  lapse rate: {y_test.mean():.3f}")

X_train: (3982, 37), X_test: (996, 37)
y_train lapse rate: 0.196
y_test  lapse rate: 0.196


## Step 12: Feature Scaling (StandardScaler)

Fit a `StandardScaler` on the training set and transform both train and test sets.
Remember: fit on train only, transform both — fitting on full data is data leakage.

In [17]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

## Bonus: K-Means Clustering

**Optional**: Use K-Means to create customer segments as an additional feature.
Use 3 clusters on scaled behavioral features (purchase_count, avg_basket_value,
days_since_last_purchase, product_categories_bought).

**Task**: Fit K-Means, add cluster labels as a feature to both train and test.

In [18]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
behavioral_cols = ["purchase_count", "avg_basket_value", "days_since_last_purchase", "product_categories_bought"]
kmeans.fit(X_train_scaled[behavioral_cols])
X_train_scaled["cluster"] = kmeans.labels_
X_test_scaled["cluster"] = kmeans.predict(X_test_scaled[behavioral_cols])

## Save Checkpoint for MP3

In [19]:
import pickle

CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Split gender_series to align with train/test
gender_train = gender_series.loc[X_train.index]
gender_test = gender_series.loc[X_test.index]

checkpoint = {
    "X_train": X_train_scaled if "X_train_scaled" in dir() else X_train,
    "X_test": X_test_scaled if "X_test_scaled" in dir() else X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_names": list(df.columns),
    "gender_train": gender_train,
    "gender_test": gender_test,
}

with open(CHECKPOINT_DIR / "mp2_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)
print(f"Checkpoint saved: {CHECKPOINT_DIR / 'mp2_checkpoint.pkl'}")
print(f"Keys: {list(checkpoint.keys())}")

Checkpoint saved: ../checkpoints/mp2_checkpoint.pkl
Keys: ['X_train', 'X_test', 'y_train', 'y_test', 'feature_names', 'gender_train', 'gender_test']


---
*End of MP2 — Continue to MP3: Baseline Modeling & Algorithm Comparison*